# 17 · Teşhis: çıkarım boru hattı altın vakalarda eğitimdeki skoru veriyor mu?

**Soru.** Raptor-80 modellerimiz eğitim sırasında 58 altın vakada ~0.92 macro-AUC alıyor (corpus görüntüleri
üzerinden). Gizli testte ise görüntüler `infer_knee.py` ile DICOM'dan yeniden üretiliyor ve LB'de model public
blend'i düşürüyor (19 Ekim: %10 ağırlık 0.958 -> 0.956). İki ön işleme farklıysa gold ölçümü bunu göremez.

**Yöntem.** Altın vakalardan sahte bir test kökü kurulur (`test.csv` = 58 vaka, `test_series` = eğitim DICOM'ları)
ve submission'daki ayarlarla `infer_knee.py` çalıştırılır. Çıkan AUC eğitimdeki gold AUC ile karşılaştırılır.

In [ ]:
import os, json, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd

def find_comp():
    for b in ["/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection"]:
        if os.path.exists(b + "/train.csv"):
            return Path(b)
    raise FileNotFoundError("competition root")
def _find(name):
    parts = Path(name).parts
    for d, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = sorted(x for x in dirs if x not in ("train_series", "test_series"))
        cand = Path(d).joinpath(*parts)
        if cand.exists():
            return cand
    return None

COMP = find_comp()
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = pd.read_csv(COMP / "train.csv")
gold = train[train[LAB[0]].notna()].copy()
print("altın vaka:", len(gold))
G = Path("/kaggle/working/goldroot"); G.mkdir(exist_ok=True)
gold[["StudyInstanceUID"]].to_csv(G / "test.csv", index=False)
ser = pd.read_csv(COMP / "train_series.csv")
ser[ser.StudyInstanceUID.isin(gold.StudyInstanceUID)].to_csv(G / "test_series.csv", index=False)
if not (G / "test_series").exists():
    os.symlink(COMP / "train_series", G / "test_series")
pd.DataFrame({"StudyInstanceUID": gold.StudyInstanceUID, **{c: 0.5 for c in LAB}}).to_csv(G / "sample_submission.csv", index=False)
print(sorted(os.listdir(G)))

In [ ]:
# Submission'daki (10s*) Raptor-80 ayarlarının aynısı
GEOM = {"SLOTS": [["Sagittal", 1, 22], ["Sagittal", 0, 18], ["Coronal", 1, 15], ["Coronal", 0, 10], ["Axial", -1, 15]],
        "IMG": 336, "CROP_MM": 140.0, "SPAN": [0.02, 0.98], "K_EVAL": 78}
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
MODELS = ["raptor_ft_kadirov80_lbl3_s101.pt", "raptor_ft_kadirov80_lbl3_coatnet_s505.pt"]
WORK = Path("/kaggle/working")
res = {}
for m in MODELS:
    p = _find(m)
    if p is None:
        print("yok:", m); continue
    out = WORK / f"gold_pred_{Path(m).stem}.csv"
    cfg = dict(GEOM, NORM="imagenet", OUT_PATH=str(out),
               ARMS=[{"file": p.name, "arch": ARCH, "res": 384, "w": 1.0, "norm": "imagenet"}])
    (WORK / "cfg.json").write_text(json.dumps(cfg))
    t = time.time()
    rc = subprocess.run([sys.executable, "-u", str(_find("third_party/raptor/infer_knee.py"))], cwd=str(WORK),
                        env=dict(os.environ, RAPTOR_CFG=str(WORK / "cfg.json"), RAPTOR_TEST_ROOT=str(G))).returncode
    print(m, "rc", rc, f"{time.time() - t:.0f}s")
    res[m] = out

In [ ]:
from sklearn.metrics import roc_auc_score
y = gold.set_index("StudyInstanceUID")[LAB].astype(float)
rows = {}
for m, out in res.items():
    if not out.exists():
        print("çıktı yok:", m); continue
    p = pd.read_csv(out).set_index("StudyInstanceUID").loc[y.index, LAB]
    per = {c: roc_auc_score(y[c], p[c]) for c in LAB if 0 < y[c].sum() < len(y)}
    rows[m] = dict(per, macro=np.mean(list(per.values())), sabit_sutun=int((p.nunique() <= 1).sum()))
tab = pd.DataFrame(rows).T.round(4)
print(tab[["macro", "sabit_sutun"]])
print("Eğitimdeki gold (corpus): s101 0.9205 | s505 0.9216")
tab